# Fake News detections.

In [248]:
# import
import os
import re
import csv
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from io import StringIO
from urllib.request import urlopen

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

RANDOM_STATE = 42

plt.rcParams["figure.figsize"] = (8, 5)
sns.set_theme(style="whitegrid")

In [249]:
# Data path
true_path = r'D:\2 - (25 - 26)\ML\FinalML\Data\True.csv'
fake_path = r'D:\2 - (25 - 26)\ML\FinalML\Data\Fake.csv'

In [250]:
# Read csv
data_true = pd.read_csv(true_path)
data_fake = pd.read_csv(fake_path)

In [251]:
data_true['class'] = 1
data_fake['class'] = 0

In [252]:
data = pd.concat([data_true, data_fake], axis= 0)
data.sample(5)

,title,text,subject,date,class
17818,"ACTRESS ACCUSES Weinstein Buddy, Actor GEORGE ...","The Hill During a recent interview, Clooney ...",left-news,"Oct 13, 2017",0
644,Senate committee questions Trump's nuclear aut...,WASHINGTON (Reuters) - A U.S. Senate committee...,politicsNews,"November 14, 2017",1
7254,Right-Wing Bigots Chased Away By Police In Br...,This is how Brussels deals with right-wing ext...,News,"March 27, 2016",0
15537,Gunmen fire shots at Greece's socialist party ...,ATHENS (Reuters) - Two gunmen fired a round of...,worldnews,"November 6, 2017",1
843,Fox News Just Shared A Poll That Trump Is Goi...,"On Sunday, Fox News had a rare moment of clari...",News,"July 16, 2017",0


In [253]:
# Information about data
print("Thông tin dữ liệu:")
display(data.info())

print("\n5 dòng đầu:")
display(data.head())

print("\nTên cột:")
print(data.columns.tolist())

print("\nSố lượng giá trị thiếu:")
print(data.isna().sum())


Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
Index: 44898 entries, 0 to 23480
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44898 non-null  object
 2   subject  44898 non-null  object
 3   date     44898 non-null  object
 4   class    44898 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 2.1+ MB


None


5 dòng đầu:


,title,text,subject,date,class
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1



Tên cột:
['title', 'text', 'subject', 'date', 'class']

Số lượng giá trị thiếu:
title      0
text       0
subject    0
date       0
class      0
dtype: int64


In [254]:
# pre clean data
data['content'] = data['title'] + ' ' +data['text']
data.drop(['title', 'subject', 'date', 'text'], axis=1, inplace=True)
data.sample(5)

,class,content
3060,0,George Takei Just F*cking HUMILIATED Trump Ov...
4024,1,Number of U.S. visas to citizens of Trump trav...
10251,1,Colombia's FARC rebels to meet Kerry in Cuba d...
3687,0,WATCH: Donald Trump Is Considering Gutting Me...
22509,0,PROPAGANDA: Star Trek Beyond – Social Justice ...


In [255]:

data = data.drop_duplicates(subset=['content']).reset_index(drop=True)

print("Số dòng sau khi xóa trùng:", len(data))

Số dòng sau khi xóa trùng: 39105


In [256]:
data.reset_index(inplace= True)
data.sample(5)

,index,class,content
26862,26862,0,Trump’s $45 Million Loan Is The Latest Way He...
23430,23430,0,WATCH: Trump Loves The GOP Healthcare Bill Bu...
10670,10670,1,California lawmaker aims to reduce eating diso...
28443,28443,0,"This Is Clinton’s Supreme Court Plan, And It ..."
26004,26004,0,There Was Something Weird About Trump’s Hair ...


In [257]:
data.drop(['index'], axis=1, inplace= True)
data.sample(5)

,class,content
32982,0,LIST OF 20 “Vetted” Refugees Who Were Charged ...
32767,0,"INDIAN-AMERICAN, Inventor Of Email Announces R..."
37427,0,LAND GRAB ALERT: Texas Rancher Could Lose 600 ...
14915,1,Indonesia warns of tough response after Papuan...
30389,0,KARMA: Race-Obsessed Detroit Free Press Editor...


In [258]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text)
    
    text = text.lower()
    
    text = re.sub(r'http\S+|www\S+', '', text)
    
    text = re.sub(r'\S+@\S+', '', text)
    
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokens = text.split()
    
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]
    
    return ' '.join(tokens)



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [259]:
data["content"] = data["content"].apply(preprocess_text)

In [260]:
data.sample(5)

,class,content
3729,1,u industry seek faster permit simpler rule tru...
14129,1,zimbabwe mnangagwa add call mugabe go harare r...
6518,1,u judge block transgender abortionrelated obam...
23106,0,watch republican senator lash cnn host called ...
25230,0,democrat sue republican plot rig election demo...


In [261]:
# slplit train and test.
x = data['content']
y = data['class']
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size= 0.25,
    random_state= 42, stratify=y
)

In [262]:
label_names = {
    1 : 'True New',
    0 : 'Fake New'
}
print("Số mẫu train:", len(x_train))
print("Số mẫu test:", len(x_test))

print("\nPhân bố nhãn train:")
print(y_train.value_counts().sort_index().rename(index=label_names))
 
print("\nPhân bố nhãn test:")
print(y_test.value_counts().sort_index().rename(index=label_names))

Số mẫu train: 29328
Số mẫu test: 9777

Phân bố nhãn train:
class
Fake New    13431
True New    15897
Name: count, dtype: int64

Phân bố nhãn test:
class
Fake New    4477
True New    5300
Name: count, dtype: int64


In [263]:
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features= 20000,
        ngram_range= (1, 2),
        min_df=2, 
        max_df=0.95,
        sublinear_tf= True
    )),
    ('svm', LinearSVC(
        C=1.0,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])
svm_pipeline.fit(x_train, y_train)

y_pred_svm = svm_pipeline.predict(x_test)

print("Result of SVM Linear:")
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=['Fake', 'Real'],
    zero_division= 0,
    digits= 6
))

Result of SVM Linear:
              precision    recall  f1-score   support

        Fake   0.997985  0.995533  0.996757      4477
        Real   0.996234  0.998302  0.997267      5300

    accuracy                       0.997034      9777
   macro avg   0.997110  0.996917  0.997012      9777
weighted avg   0.997036  0.997034  0.997034      9777



# Otimazing Model SVM with GridSearchCV